# Phase 1: Data Scouting & Cleaning

## Overview: The Prep-Work
In this notebook, we transform raw, noisy news data into a structured numerical format that a computer can understand.

### The Kitchen Preparation (Analogies)
*   **For the 5-year-old**: Imagine you are making a soup. You can't just throw the dirt and the peels of the potatoes into the pot. You have to wash them, peel them, and cut them into small pieces. That is what we are doing with our words!


*   **For the Senior Executive**: Data quality is the single greatest determinant of model ROI. We are implementing a 'Garbage In, Garbage Out' guardrail by standardizing the text features to ensure the classifier focuses on meaningful semantic patterns rather than typographical noise.

In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
import nltk
import re
import string

# Colab Integration: Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive/Project 2/project-nlp-challenge'
    print("✅ Running in Colab. Google Drive mounted.")
except ImportError:
    BASE_PATH = '.'
    print("💻 Running Locally.")

# NLTK Downloads for ephemeral environments
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

# Ensure directories exist
os.makedirs(os.path.join(BASE_PATH, 'models'), exist_ok=True)
os.makedirs(os.path.join(BASE_PATH, 'dataset'), exist_ok=True)


Mounted at /content/drive
✅ Running in Colab. Google Drive mounted.


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


## 1. Data Ingestion
Load our primary dataset. It contains nearly 40,000 articles, balanced almost 50/50 between Real (1) and Fake (0) news.

In [ ]:

data_path = os.path.join(BASE_PATH, 'dataset/data.csv')
df = pd.read_csv(data_path)
df.head(5)

,label,title,text,subject,date
0,1,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,1,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,1,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,1,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


## 2. Fusion (Title + Text)

### The Strategy: Why combine Title and Body?
Created a new column `full_text` by concatenating `title` and `text`.

**Why this is better than keeping them separate (for now):**
1.  **Context Density**: In fake news, the 'hook' is often in the title (clickbait) while the body supports it with misinformation. By merging them, we ensure our model doesn't miss the correlation between a sensationalized headline and the actual content.

2.  **Handling Sparse Data**: Some articles have very short bodies. Including the title guarantees that every record has a minimum amount of semantic data to be processed.

3.  **Baseline Simplicity**: As our first model, we want a 'holistic' view of the article. Separating them would double our feature count (title-features vs body-features), which might lead to overfitting on sparse title words.

In [ ]:
df['full_text'] = df['title'] + " " + df['text']
print("✅ 'full_text' Column created.")
df.head(10)

✅ 'full_text' Column created.


,label,title,text,subject,date,full_text
0,1,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017","As U.S. budget fight looms, Republicans flip t..."
1,1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",U.S. military to accept transgender recruits o...
2,1,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",Senior U.S. Republican senator: 'Let Mr. Muell...
3,1,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",FBI Russia probe helped by Australian diplomat...
4,1,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",Trump wants Postal Service to charge 'much mor...
5,1,"White House, Congress prepare for talks on spe...","WEST PALM BEACH, Fla./WASHINGTON (Reuters) - T...",politicsNews,"December 29, 2017","White House, Congress prepare for talks on spe..."
6,1,"Trump says Russia probe will be fair, but time...","WEST PALM BEACH, Fla (Reuters) - President Don...",politicsNews,"December 29, 2017","Trump says Russia probe will be fair, but time..."
7,1,Factbox: Trump on Twitter (Dec 29) - Approval ...,The following statements were posted to the ve...,politicsNews,"December 29, 2017",Factbox: Trump on Twitter (Dec 29) - Approval ...
8,1,Trump on Twitter (Dec 28) - Global Warming,The following statements were posted to the ve...,politicsNews,"December 29, 2017",Trump on Twitter (Dec 28) - Global Warming The...
9,1,Alabama official to certify Senator-elect Jone...,WASHINGTON (Reuters) - Alabama Secretary of St...,politicsNews,"December 28, 2017",Alabama official to certify Senator-elect Jone...


## 3. The Scrubbing (Cleaning Logic)

### 🛠 Why these cleaning steps matter?
1.  **Lowercase**: Computers see 'Apple' and 'apple' as different entities. Lowercasing collapses them into the same concept, reducing the 'vocabulary' size without losing meaning.
2.  **Punctuation Removal**: Symbols like '!!!' or '???' carry emotional weight but can confuse a baseline classifier. We strip them to focus on the CORE vocabulary.
3.  **Tokenization**: This is the act of 'slicing the potato'. We break the string into individual units (tokens) so the computer can count them.
4.  **Stopword Removal**: Words like 'the', 'is', 'and' appear in almost every sentence. Because they don't help us distinguish between Real and Fake news (they are neutral), we remove them to boost the 'signal' of unique words like 'conspiracy', 'evidence', or 'verified'.

In [ ]:
def clean_text(text):
    """
    Standard Cleaning logic:
    """
    if pd.isna(text):
        return ""

    # 1. Lowercase
    text = text.lower()

    # 2. Remove punctuation and symbols
    import re, string
    pattern = re.compile('[%s]' % re.escape(string.punctuation))
    text = pattern.sub('', text)

    # 3. Tokenize
    from nltk.tokenize import word_tokenize
    tokens = word_tokenize(text)

    # 4. Remove Stopwords
    from nltk.corpus import stopwords
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [w for w in tokens if w not in stop_words]

    return " ".join(filtered_tokens)


In [ ]:
# Apply the cleaning (This creates the 'cleaned_text' column)

print("Scrubbing data...")
df['cleaned_text'] = df['full_text'].apply(clean_text)
print("✅ Scrubbing Complete.")


Scrubbing data...
✅ Scrubbing Complete.


## 4. Vectorization (The Flavor Embeddings)
We use **TF-IDF (Term Frequency-Inverse Document Frequency)**.

### 🧠 Why TF-IDF instead of simple counting?
Simple counting rewards a word just for being frequent. **TF-IDF is smarter**: it rewards a word if it is frequent in *one* article but **rare** in the entire database. This highlights truly unique keywords that define 'Fake News' patterns vs 'Real news' patterns.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

# We limit to 5000 features to maintain transparency and avoid 'learning by heart' (overfitting)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
print("✅ Vectorizer Initialized: TF-IDF with Unigrams and Bigrams.")

✅ Vectorizer Initialized: TF-IDF with Unigrams and Bigrams.


## 5. The Final Slice (Splitting the Data)

### 🧪 Why split?
If you want to know if a student actually learned math, you don't give them the same problems they saw in class for the final exam.
*   **Train Set (80%)**: The 'Classroom examples'.
*   **Test Set (20%)**: The 'Final Exam' (data the model has never seen).

We use **Stratification** to ensure that both the Train and Test sets have the same percentage of Real vs Fake news as the original data.

In [ ]:
from sklearn.model_selection import train_test_split

# Separating for visualization/demo in the next phase
train_df, test_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df['label'])

print(f"🎓 Training Set: {len(train_df)} specimens")
print(f"📝 Testing Set: {len(test_df)} specimens")

🎓 Training Set: 31953 specimens
📝 Testing Set: 7989 specimens


## 6. Persisting Data & Preparing Ingredients
We save our progress to ensure Phase 2 starts with all necessary ingredients ready.


In [ ]:
# 1. Re-split after cleaning to ensure cleaned_text is in train/test sets
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df['label'])

# 2. Fit & Save Vectorizer (Important: Fit on cleaned text!)
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
tfidf.fit(train_df['cleaned_text'].astype(str))
joblib.dump(tfidf, os.path.join(BASE_PATH, 'models/vectorizer.joblib'))

# 3. Save Final Datasets
train_df.to_csv(os.path.join(BASE_PATH, 'dataset/train.csv'), index=False)
test_df.to_csv(os.path.join(BASE_PATH, 'dataset/test.csv'), index=False)
df.to_csv(os.path.join(BASE_PATH, 'dataset/cleaned_data.csv'), index=False)
print("✅ Progress Saved: Ingredients ready in models/ and dataset/")


✅ Progress Saved: Ingredients ready in models/ and dataset/
